In [1]:
!pip install openpyxl


In [2]:
# Importing required library ...
import pandas as pd
import numpy as np 
#import utilities
import datetime
import time
#import xgboost
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
#import preprocess
from sklearn.model_selection import KFold
import openpyxl
#import xlrd


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

from rdkit.Chem import MolStandardize

import joblib

import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem

from rdkit.Chem import Descriptors,rdMolDescriptors


In [4]:
import pandas as pd 

In [4]:
### Collecting the data from different sources

data1=pd.read_csv('data/water_solubility_data.csv') ### bnn lab data 
data2=pd.read_csv('data/data_paper.csv',encoding='latin1') ### recent  paper data 
#data3=pd.read_csv('data/VCC_data1.txt',sep=" ",header=None)### 6 VCC DATA 1311
#data4=pd.read_csv('data/new222.csv') ##  data from github  2008
data5 = pd.read_excel('data/Supplementary_data.xlsx')
data6=pd.read_csv('data/dataset-not-FA.csv') ### bnn lab data    curated data used in Sorkun paper
#data7 = pd.read_excel('new_data.xls')
data7=pd.read_csv('data/Sumeen_data_water.csv') ### Sumeen Lee data (multi-solvent paper)
#### Test Data 
test_set=pd.read_csv('data/dataset-E.csv')



In [5]:
data7

,Unnamed: 0,Solute SMILES,LogS
0,0,Cc1c3c(c(c2c1cccc2)C)cccc3,-6.570001
1,1,S(=O)(=O)(NC(=O)OC)c1ccc(cc1)N,-1.660000
2,2,O(C(CCO[N+](=O)[O-])C)[N+](=O)[O-],-1.659000
3,3,c1(ccc(cc1)OCC)N(C(=O)N)C,-1.658000
4,4,c1(ccccc1)C(C(=O)NC(c2ccccc2)C)N,-1.658000
...,...,...,...
23782,23782,ON=C1NC=NC2=C1NC=N2,-2.399523
23783,23783,CC=CCCl,-1.954357
23784,23784,[AsH2]CC,-2.911266
23785,23785,CCCCC(CC)C(NC(N)=O)=O,-2.667772


In [7]:
### In order to merge the data all the dataset should have identical column name ... 
data1=data1[['Smiles','LogS']]
## Changing the column name to be the identical with other dataset 
data1=data1.rename(columns={ "Smiles": "SMILES","LogS":"Solubility"})

### Selecting the requiered column 
data2=data2[['SMILES','LogS']]
## Changing the column name to be the identical with other dataset 
data2=data2.rename(columns={ "LogS":"Solubility"})#

### Changing the column name ...

data5=data5[['SMILES','logS']]
data5 = data5.rename(columns={'logS': 'Solubility'})

data6=data6[['SMILES','Solubility']]


#data7=data7[['Smiles','logS(uni)']]
#data7=data7.rename(columns={ "Smiles": "SMILES","logS(uni)":"Solubility"})


KeyError: "None of [Index(['Smiles', 'LogS'], dtype='object')] are in the [columns]"

In [8]:
data7=data5[['Solute SMILES','LogS']]
data7=data7.rename(columns={ "Solute SMILES":"SMILES","LogS":"Solubility"})

KeyError: "None of [Index(['Solute SMILES', 'LogS'], dtype='object')] are in the [columns]"

In [33]:
#data1.insert(0, 'New_ID', range('WSD', 'WSD' + len(data1)))
#data1
#



In [13]:
#data7.loc[:, 'Solute SMILES','LogS']
#data7 = data7.iloc[:, -2:]
data7=data7.rename(columns={ "Solute SMILES":"SMILES","LogS":"Solubility"})


In [14]:
data7

,SMILES,Solubility
0,Cc1c3c(c(c2c1cccc2)C)cccc3,-6.570001
1,S(=O)(=O)(NC(=O)OC)c1ccc(cc1)N,-1.660000
2,O(C(CCO[N+](=O)[O-])C)[N+](=O)[O-],-1.659000
3,c1(ccc(cc1)OCC)N(C(=O)N)C,-1.658000
4,c1(ccccc1)C(C(=O)NC(c2ccccc2)C)N,-1.658000
...,...,...
23782,ON=C1NC=NC2=C1NC=N2,-2.399523
23783,CC=CCCl,-1.954357
23784,[AsH2]CC,-2.911266
23785,CCCCC(CC)C(NC(N)=O)=O,-2.667772


In [16]:
### Merging the train dataset to one dataframe 
frames = [data1,data2,data5,data6,data7]
train_set = pd.concat(frames)
### Size of the train and test dataset before preprocess....
print('Size of the train dataset :',len(train_set))
print('Size of the test dataset :',len(test_set))


train_set.to_csv('data/train4_final.csv')
test_set.to_csv('data/test_final.csv')

Size of the train dataset : 52646
Size of the test dataset : 1291


In [17]:
train_set.reset_index(drop=True)
test_set.reset_index(drop=True)

,ID,Name,InChI,InChIKey,SMILES,Solubility
0,E-1,n-pentane,"InChI=1S/C5H12/c1-3-5-4-2/h3-5H2,1-2H3",OFBQJSOFQDEBGM-UHFFFAOYSA-N,CCCCC,-3.18
1,E-2,cyclopentane,InChI=1S/C5H10/c1-2-4-5-3-1/h1-5H2,RGSFGYAAUTVSQA-UHFFFAOYSA-N,C1CCCC1,-2.64
2,E-3,n-hexane,"InChI=1S/C6H14/c1-3-5-6-4-2/h3-6H2,1-2H3",VLKZOEOYAKHREP-UHFFFAOYSA-N,CCCCCC,-3.84
3,E-4,2-methylpentane,"InChI=1S/C6H14/c1-4-5-6(2)3/h6H,4-5H2,1-3H3",AFABGHUZZDYHJO-UHFFFAOYSA-N,CCCC(C)C,-3.74
4,E-5,"2,2-dimethylbutane","InChI=1S/C6H14/c1-5-6(2,3)4/h5H2,1-4H3",HNRMPXKDFBEGFZ-UHFFFAOYSA-N,CCC(C)(C)C,-3.55
...,...,...,...,...,...,...
1286,E-1287,malathion,InChI=1S/C10H19O6PS2/c1-5-15-9(11)7-8(10(12)16...,JXSJBGJIGXNWCI-UHFFFAOYSA-N,CCOC(=O)CC(SP(=S)(OC)OC)C(=O)OCC,-3.37
1287,E-1288,chlorpyriphos,"InChI=1S/C9H11Cl3NO3PS/c1-3-14-17(18,15-4-2)16...",SBPBAQFWLVIOKP-UHFFFAOYSA-N,CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl,-5.49
1288,E-1289,prostaglandin_E2,InChI=1S/C20H32O5/c1-2-3-6-9-15(21)12-13-17-16...,XEYBRNLFEZDVAW-UHFFFAOYSA-N,CCCCCC(O)C=CC1C(O)CC(=O)C1CC=CCCCC(=O)O,-2.47
1289,E-1290,"p,p'-DDT",InChI=1S/C14H9Cl5/c15-11-5-1-9(2-6-11)13(14(17...,YVGGHNCTFXOJCH-UHFFFAOYSA-N,c(ccc(c1)Cl)(c1)C(c(ccc(c2)Cl)c2)C(Cl)(Cl)Cl,-7.15


In [18]:
from rdkit.Chem import MolFromSmiles as smi2mol
from rdkit.Chem import MolToSmiles as mol2smi

## Function to create canonical smiles 
def canon(smi):
    try:
        mol=smi2mol(smi, sanitize=True)
        smi_canon=mol2smi(mol, isomericSmiles=False, canonical=True)
        return(smi_canon)
    except:
        print("ERROR")
        return(smi)

In [19]:
#### Applying function to create the column with canonical smiles.  
train_set['smiles_canon'] = [canon(smi) for smi in train_set.SMILES]
test_set['smiles_canon'] = [canon(smi) for smi in test_set.SMILES]

[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:31] WARNING: not removing hydrogen atom without neighbors
[22:32:32] WARNING: not removing hydrogen atom without neighbors
[22:32:32] WARNING: not removing hydrogen atom without neighbors
[22:32:32] WARNING: not removing hydrogen atom without neighbors
[22:32:32] WARNING: not r

In [20]:
### Calculate  the occurence of  smiles in both train and test dataset....
train_set['occurence'] = train_set.groupby('smiles_canon')['smiles_canon'].transform('count')
test_set['occurence'] = test_set.groupby('smiles_canon')['smiles_canon'].transform('count')

In [21]:
### Taking out Unique smiles from the train and test dataset 
train_set1= train_set[train_set['occurence']==1]
print(train_set1.shape)
test_set1= test_set[test_set['occurence']==1]
print(test_set1.shape)

(11611, 4)
(1273, 8)


In [22]:
### Selecting specific column of the dataframe ..
train_set1=train_set1[['smiles_canon','Solubility','occurence']]
test_set1=test_set1[['smiles_canon','Solubility','occurence']]

In [23]:
#### Taking out the dataframe which has duplicate smiles means more than one time occurence in the dataset ...
train_set2= train_set[train_set['occurence']>1]
print(train_set2.shape)
test_set2= test_set[test_set['occurence']>1]
print(test_set2.shape)

(41035, 4)
(18, 8)


In [24]:
#Extract duplicate rows
id1 = train_set2["smiles_canon"]
train_set2=train_set2[id1.isin(id1[id1.duplicated()])].sort_values("smiles_canon")
id2 = test_set2["smiles_canon"]
test_set2=test_set2[id2.isin(id2[id2.duplicated()])].sort_values("smiles_canon")


In [25]:
train_set2.shape

(41035, 4)

In [26]:
train_set2

,SMILES,Solubility,smiles_canon,occurence
3059,B#N,-6.394784,B#N,3
14633,B#N,-6.394784,B#N,3
22577,B#N,-6.394784,B#N,3
8155,B12B3B4B1C234,-4.742403,B12B3B4B1C234,2
18823,B12B3B4B1C234,-4.742403,B12B3B4B1C234,2
...,...,...,...,...
4183,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
23189,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
7011,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
17845,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5


In [27]:
train_set2.reset_index(drop=True)

,SMILES,Solubility,smiles_canon,occurence
0,B#N,-6.394784,B#N,3
1,B#N,-6.394784,B#N,3
2,B#N,-6.394784,B#N,3
3,B12B3B4B1C234,-4.742403,B12B3B4B1C234,2
4,B12B3B4B1C234,-4.742403,B12B3B4B1C234,2
...,...,...,...,...
41030,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
41031,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
41032,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5
41033,[nH]1nc2cncnc2n1,-0.231900,c1ncc2n[nH]nc2n1,5


In [28]:
train_set2=train_set2[['smiles_canon','Solubility','occurence']]
test_set2=test_set2[['smiles_canon','Solubility','occurence']]

In [29]:
train_set2.shape

(41035, 3)

In [30]:
train_set2

,smiles_canon,Solubility,occurence
3059,B#N,-6.394784,3
14633,B#N,-6.394784,3
22577,B#N,-6.394784,3
8155,B12B3B4B1C234,-4.742403,2
18823,B12B3B4B1C234,-4.742403,2
...,...,...,...
4183,c1ncc2n[nH]nc2n1,-0.231900,5
23189,c1ncc2n[nH]nc2n1,-0.231900,5
7011,c1ncc2n[nH]nc2n1,-0.231900,5
17845,c1ncc2n[nH]nc2n1,-0.231900,5


In [35]:
threshold = 0.5  # Adjust the threshold as needed
#df_filtered_train = train_set2.groupby('smiles_canon').filter(lambda x: x['Solubility'].max() - x['Solubility'].min() <= threshold)
df_difference_train_bal = train_set2.groupby('smiles_canon').filter(lambda x: x['Solubility'].max() - x['Solubility'].min() > threshold)



In [36]:
threshold = 0.5  # Adjust the threshold as needed
df_filtered_test = test_set2.groupby('smiles_canon').filter(lambda x: x['Solubility'].max() - x['Solubility'].min() <= threshold)



In [33]:
#Extract duplicate rows
id1 = train_set2["smiles_canon"]
train_set2=train_set2[id1.isin(id1[id1.duplicated()])].sort_values("smiles_canon")
id2 = test_set2["smiles_canon"]
test_set2=test_set2[id2.isin(id2[id2.duplicated()])].sort_values("smiles_canon")


In [38]:
df_filtered_train.shape
df_filtered_test.shape

NameError: name 'df_filtered_train' is not defined

In [33]:
df_filtered_train

,SMILES,Solubility,smiles_canon,occurence
1089,CN(CC/C=C/1\c2ccccc2Sc2c1cc(F)cc2)C.Br,-5.603798,Br.CN(C)CCC=C1c2ccccc2Sc2ccc(F)cc21,2
1090,CN(CC/C=C\1/c2ccccc2Sc2c1cc(F)cc2)C.Br,-5.603798,Br.CN(C)CCC=C1c2ccccc2Sc2ccc(F)cc21,2
1791,Oc1ccc2c(c1)[C@]13CCCC[C@H]3[C@@H](C2)N(CC1)CC...,-4.833904,Br.Oc1ccc2c(c1)C13CCCCC1C(C2)N(CCc1ccccc1)CC3,2
1792,Oc1ccc2c(c1)[C@@]13CCCC[C@@H]3[C@H](C2)N(CC1)C...,-4.833904,Br.Oc1ccc2c(c1)C13CCCCC1C(C2)N(CCc1ccccc1)CC3,2
69,BrC(Br)(Br)Br,-3.140000,BrC(Br)(Br)Br,3
...,...,...,...,...
9794,c1ncn[nH]1,0.788505,c1nc[nH]n1,2
1642,[NH]1C=NC2=C1C=NC=N2,0.620000,c1ncc2[nH]cnc2n1,2
2547,C1=C2C(=NC=N1)N=CN2,0.619391,c1ncc2[nH]cnc2n1,2
4507,C1=NC=NC2=NNN=C21,-0.231900,c1ncc2n[nH]nc2n1,2


In [39]:
### calculate mean of all the solubility values given by duplicate smiles and make it unique 

train_set3=df_filtered_train.groupby('smiles_canon').mean().reset_index()
print(train_set3.shape)
test_set3=df_filtered_test.groupby('smiles_canon').mean().reset_index()
print(test_set3.shape)

NameError: name 'df_filtered_train' is not defined

In [40]:
df_difference_train_bal

,smiles_canon,Solubility,occurence
10423,Brc1ccc(Oc2cc(Br)c(Br)cc2Br)c(Br)c1,-2.367686,3
1507,Brc1ccc(Oc2cc(Br)c(Br)cc2Br)c(Br)c1,-8.371599,3
20814,Brc1ccc(Oc2cc(Br)c(Br)cc2Br)c(Br)c1,-2.367686,3
8222,C,-0.900000,6
2870,C,-0.900000,6
...,...,...,...
2321,c1ccc2c(c1)sc1ccccc12,-4.380000,5
17244,c1cn[nH]c1,1.287890,4
1695,c1cn[nH]c1,-0.545218,4
4184,c1cn[nH]c1,-0.545200,4


In [41]:
test_set2.shape

(18, 3)

In [42]:
train_set2 = train_set2.join(
    train_set2.groupby('smiles_canon')[['Solubility']].transform('mean')  # Calculate the mean for each group
        .rename(columns='mean {} '.format)  # Rename columns 
)

In [43]:
train_set2.shape

(98503, 4)

In [44]:
train_set2

,smiles_canon,Solubility,occurence,mean Solubility
0,CC(C)C,-3.075906,5,-2.655181
0,CC(C)C,-3.075906,5,-3.196329
0,CC(C)C,-3.075906,5,-12.868622
0,CC(C)C,-3.075906,5,-4.021333
0,CC(C)C,-3.075906,5,-6.569382
...,...,...,...,...
23782,ON=c1[nH]cnc2nc[nH]c12,-2.399523,3,-2.399523
23783,CC=CCCl,-1.954357,6,-1.955645
23784,CC[AsH2],-2.911266,4,-2.911299
23785,CCCCC(CC)C(=O)NC(N)=O,-2.667772,4,-2.667844


In [45]:

test_set2 = test_set2.join(
    test_set2.groupby('smiles_canon')[['Solubility']]
        .transform('mean')  # Calculate the mean for each group
        .rename(columns='mean {} '.format)  # Rename columns 
)


In [46]:
train_set2.shape

(98503, 4)

In [47]:
test_set2.head(10)

,smiles_canon,Solubility,occurence,mean Solubility
402,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,-4.30,2,-4.150
1003,CC(=O)OCC(=O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3C(...,-4.00,2,-4.150
665,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,-3.64,2,-3.730
664,CC(CCC(=O)O)C1CCC2C3C(O)CC4CC(O)CCC4(C)C3CCC12C,-3.82,2,-3.730
401,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,-3.64,2,-3.705
413,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,-3.77,2,-3.705
1224,CC1CCCCC1C,-4.27,2,-4.300
827,CC1CCCCC1C,-4.33,2,-4.300
1081,CCC(C)(O)C(C)C,-1.22,2,-1.035
256,CCC(C)(O)C(C)C,-0.85,2,-1.035


In [48]:
### calculate mean of all the solubility values given by duplicate smiles and make it unique 

train_set3=train_set2.groupby('smiles_canon').mean().reset_index()
print(train_set3.shape)
test_set3=test_set2.groupby('smiles_canon').mean().reset_index()
print(test_set3.shape)

(11251, 4)
(9, 4)


In [49]:
### Combining the dataframe with occurence 1 and more than one with mean of the solubility
train_set= pd.concat([train_set1,train_set3],axis=0)
test_set= pd.concat([test_set1,test_set3],axis=0)

In [50]:
print(train_set.shape)
print(test_set.shape)

(22862, 4)
(1282, 4)


In [24]:
# savinf the combine 6 dataset 
train_set.to_csv('data/combined_data6.csv')

In [51]:
### Remove the smiles from train dataset with matching test dataset...
smiles_train_canon=train_set.smiles_canon
smiles_test_canon=test_set.smiles_canon

In [52]:
overlap1 = 0
for x in smiles_train_canon:
    if x in smiles_test_canon:
        overlap1+=1
print("%i of the train molecules are in the test set"%(overlap1))

overlap2 = 0
for x in smiles_test_canon:
    if x in smiles_train_canon:
        overlap2+=1
print("%i of the test molecules are in the train set"%(overlap2))

0 of the train molecules are in the test set
0 of the test molecules are in the train set


In [53]:
Match_rows = pd.merge(test_set, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows))

1282


In [54]:
### Finding the matching smiles in the train dataset..
cond = train_set['smiles_canon'].isin(Match_rows['smiles_canon'])

#### Dropping the smiles which is same and making dataframe with uniques smiles and no smiles same in ttrain  dataset 
train_set.drop(train_set[cond].index, inplace = True)

In [55]:
print(train_set.shape)
print(test_set.shape)

(20299, 4)
(1282, 4)


In [56]:
### Cross check in order to find that any matching smiles exist in the train dataset with test dataset
Match_rows = pd.merge(test_set, train_set, on=['smiles_canon'], how='inner')
#mergedStuff.head()
print(len(Match_rows))

0


In [57]:
train_set.reset_index(drop=True)

,smiles_canon,Solubility,occurence,mean Solubility
0,CCCCC(O)CCC,-2.031926,1.0,NaN
1,C=CCN=C(N)S,-0.294279,1.0,NaN
2,CCCOC(=O)C(C)(C)O,-0.164918,1.0,NaN
3,CC(C)C1OC(=O)c2ccccc21,-2.256605,1.0,NaN
4,CCOC1CCC(C(=O)OCCN(CC)CC)CC1,-2.714546,1.0,NaN
...,...,...,...,...
20294,c1csnn1,0.587888,3.0,-1.637934
20295,c1nc[nH]n1,0.957535,5.0,-0.939111
20296,c1ncc2[nH]cnc2n1,0.619597,4.0,-2.040133
20297,c1ncc2cn[nH]c2n1,-1.380369,3.0,-1.217183


In [58]:
print(train_set.shape)
print(test_set.shape)

(20299, 4)
(1282, 4)


In [67]:
### Saving the data to the disk 
train_set.to_csv('data/unique_train5_new24.csv')
test_set.to_csv('data/unique_test_new24.csv')

In [68]:
data1=pd.read_csv('data/unique_train5_new24.csv')
data2=pd.read_csv('data/unique_test_new24.csv')

In [59]:
train_smiles = set(train_set['smiles_canon'])
test_smiles = set(test_set['smiles_canon'])
overlap = train_smiles.intersection(test_smiles)

if overlap:
    print("There are overlapping smiles between the training and test datasets:")
    print(overlap)
else:
    print("No overlapping smiles between the training and test datasets.")

No overlapping smiles between the training and test datasets.
